# Module 19 — Module 08 as a state machine

**THE ONE IDEA:** **the `while` loop IS the conditional edge.** LangGraph does not add a
capability — it makes the loop you already wrote *inspectable*.

Module 08 was:

```python
while True:
    r = call_model(messages)
    if finish_reason != "tool_calls": break      # <- this line
    run_tools(); append_results()
```

That one `if` becomes `add_conditional_edges`. Same answer, same tool calls, same number
of API round trips. **07, 08 and 19 all land on 10000** — that equivalence is the point
of the ladder.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json, operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from _providers import get_client
from _tools import openai_schemas, run_tool

client, MODEL, _ = get_client("openai")
TOOLS = ["search_policy", "calculate"]

class S(TypedDict):
    # The reducer. Nodes return only the CHANGE; LangGraph merges with operator.add.
    # Without Annotated, the next node's value would REPLACE the list and you would
    # lose history silently. That is the single most common LangGraph bug.
    messages: Annotated[list, operator.add]

## Two nodes and one router

A node is `(state) -> state-update`. Plain Python — no LangChain message classes, no
`langchain_openai`. The raw SDK dicts from module 08 go straight in.

In [ ]:
def agent(state: S):
    r = client.chat.completions.create(model=MODEL, max_tokens=500,
                                       tools=openai_schemas(TOOLS),
                                       messages=state["messages"])
    return {"messages": [r.choices[0].message]}          # the CHANGE, not the whole state

def tools(state: S):
    last = state["messages"][-1]
    out = []
    for tc in last.tool_calls:
        args = json.loads(tc.function.arguments)
        print(f"    tool: {tc.function.name}({args})")
        out.append({"role": "tool", "tool_call_id": tc.id,
                    "content": run_tool(tc.function.name, args)})
    return {"messages": out}

def route(state: S) -> str:
    """THIS IS THE `if` FROM MODULE 08. Nothing more."""
    last = state["messages"][-1]
    return "tools" if getattr(last, "tool_calls", None) else END

## Wire it up

In [ ]:
b = StateGraph(S)
b.add_node("agent", agent)
b.add_node("tools", tools)
b.add_edge(START, "agent")
b.add_conditional_edges("agent", route, {"tools": "tools", END: END})
b.add_edge("tools", "agent")               # the loop-back — this is the `while`
graph = b.compile()

# draw_ascii() needs grandalf; mermaid is pure Python and renders in VS Code
print(graph.get_graph().draw_mermaid())

## Run it

In [ ]:
Q = ("What is the early repayment charge in year 2 on a 250000 loan? "
     "Look up the policy, then calculate it.")
final = graph.invoke({"messages": [{"role": "user", "content": Q}]})
print("\nANSWER:", final["messages"][-1].content)
print("messages in final state:", len(final["messages"]))

## Streaming the state, step by step

`.stream()` is the payoff you cannot get from a bare `while` without writing it
yourself.

In [ ]:
for update in graph.stream({"messages": [{"role": "user", "content": Q}]},
                           stream_mode="updates"):
    for node, change in update.items():
        kinds = [getattr(m, "role", None) or m.get("role") for m in change["messages"]]
        print(f"  node={node:6} added {len(change['messages'])} message(s) {kinds}")

print("""
LESSON - the graph and the while loop are the same computation. What changed:

  the `if`        -> add_conditional_edges (named, visualisable - see the ASCII)
  the loop-back   -> add_edge("tools", "agent")
  messages += x   -> a REDUCER declared once on the state schema

What you GAINED, and none of it is free to hand-roll:
  - topology you can draw and hand to someone else
  - .stream() over state updates, for a UI or a debugger
  - the seam where checkpointing and interrupts plug in (module 20)

What you did NOT gain: any new capability. Module 08 already answered this
question, with the same tools, in the same number of round trips.

Reach for LangGraph at the point you need checkpointing, HITL or replay. Not
for the loop itself - you can write the loop.""")

---

**Next:** `20_langgraph_checkpoint_and_hitl.ipynb` — where it starts earning its keep.